# 01 — Análise Exploratória (EDA)
Dataset: harrachimustapha/hotel-reviews-dataset (Kaggle)


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from pathlib import Path

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Carregar dados

In [ ]:
# Ajuste o nome do arquivo conforme o CSV baixado do Kaggle
RAW_PATH = Path('../data/raw')
csv_files = list(RAW_PATH.glob('*.csv'))
print('Arquivos disponíveis:', csv_files)

df = pd.read_csv(csv_files[0], low_memory=False)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('Colunas:', df.columns.tolist())
df.info()

## 2. Valores nulos e duplicatas

In [ ]:
null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print('% de nulos por coluna:')
print(null_pct[null_pct > 0])

print(f'\nDuplicatas: {df.duplicated().sum()}')

## 3. Distribuição do Reviewer_Score

In [ ]:
if 'Reviewer_Score' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(df['Reviewer_Score'].dropna(), bins=20, color='steelblue', edgecolor='white')
    axes[0].set_title('Distribuição de Reviewer_Score')
    axes[0].set_xlabel('Score')
    axes[0].set_ylabel('Frequência')
    
    # Distribuição de sentimento derivado
    def score_to_sentiment(s):
        if s <= 5: return 'negativo'
        elif s <= 7: return 'neutro'
        else: return 'positivo'
    
    sent = df['Reviewer_Score'].apply(score_to_sentiment).value_counts()
    colors = {'negativo': '#EF4444', 'neutro': '#F59E0B', 'positivo': '#10B981'}
    axes[1].bar(sent.index, sent.values, color=[colors[s] for s in sent.index])
    axes[1].set_title('Sentimento derivado do Score')
    axes[1].set_ylabel('Quantidade')
    
    plt.tight_layout()
    plt.show()
    print(sent)

## 4. Comprimento dos reviews

In [ ]:
text_cols = [c for c in ['Positive_Review', 'Negative_Review', 'Review_Text', 'text'] if c in df.columns]
print('Colunas de texto encontradas:', text_cols)

for col in text_cols:
    lengths = df[col].dropna().astype(str).apply(len)
    print(f'\n{col}: média={lengths.mean():.0f} | mediana={lengths.median():.0f} | max={lengths.max()}')
    plt.figure(figsize=(8,4))
    plt.hist(lengths.clip(upper=2000), bins=50, color='steelblue', edgecolor='white')
    plt.title(f'Comprimento de {col} (até 2000 chars)')
    plt.xlabel('Caracteres')
    plt.show()

## 5. Top hotéis por volume de reviews

In [ ]:
if 'Hotel_Name' in df.columns:
    top_hotels = df['Hotel_Name'].value_counts().head(15)
    fig = px.bar(x=top_hotels.values, y=top_hotels.index, orientation='h',
                 labels={'x': 'Quantidade de Reviews', 'y': 'Hotel'},
                 title='Top 15 Hotéis por Volume de Reviews')
    fig.update_layout(height=500)
    fig.show()

## 6. Detecção de idioma (amostra)

In [ ]:
from src.data.preprocessor import detect_language, combine_reviews

sample = df.sample(min(500, len(df)), random_state=42)
sample['combined_text'] = sample.apply(combine_reviews, axis=1)
sample['language'] = sample['combined_text'].apply(detect_language)

lang_counts = sample['language'].value_counts()
print('Distribuição de idiomas (amostra de 500):')
print(lang_counts)
lang_counts.plot(kind='bar', color='steelblue')
plt.title('Idiomas detectados')
plt.show()

## 7. Conclusões da EDA
- Anotar aqui os principais achados antes de prosseguir para o preprocessamento.